# Stage 03b — Per-animal frozen clustering configuration

Point the **RUN TARGET** cell at one Stage 03a model folder (all `sim_matrix_*_open.npy` files inside). The notebook runs **every** matrix in that folder:

- **Gallery datasets** (`bristol_gorillas`, `spac_wild`, `elephants`, `gcn_id`): **2,000 Optuna trials each** (~2–3 h total on CPU for four animals; Bristol alone ~1.5 h).
- Faster smoke test: `APEXID_OPTUNA_N_TRIALS=100` before Run All, or interrupt and resume later (SQLite keeps progress).
- **Jaguar** (`jaguar_antonio`): frozen query-only defaults (no Optuna).

03a matrices are **flat** in one bucket folder:

`outputs/stage03/03a/{bucket}/sim_matrix_{descriptor}_{dataset}_open.npy`

03b writes to the matching flat bucket:

`outputs/stage03/03b/{bucket}/best_configs_{descriptor}_{dataset}.json`


In [1]:
# ============================================================
# RUN TARGET — edit these two lines, then Run All
# ============================================================
from pathlib import Path

HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE

# Flat folder with all 03a .npy files for this model family.
STAGE03A_BUCKET = "FineTuned_MiewID_DAF"

# Descriptor token in filenames: sim_matrix_{GLOBAL_DESCRIPTOR}_{dataset}_open.npy
GLOBAL_DESCRIPTOR = "ApexID_Miewid_DAF"  # must match 03a matrix filenames (Miewid not MiewID)

STAGE03A_MODEL_DIR = (REPO_ROOT / "outputs" / "stage03" / "03a" / STAGE03A_BUCKET).resolve()
STAGE03B_MODEL_DIR = (REPO_ROOT / "outputs" / "stage03" / "03b" / STAGE03A_BUCKET).resolve()
STAGE03B_MODEL_DIR.mkdir(parents=True, exist_ok=True)
MATRIX_GLOB = f"sim_matrix_{GLOBAL_DESCRIPTOR}_*_open.npy"

print("REPO_ROOT:", REPO_ROOT)
print("STAGE03A_MODEL_DIR:", STAGE03A_MODEL_DIR, "| exists:", STAGE03A_MODEL_DIR.is_dir())
print("STAGE03B_MODEL_DIR:", STAGE03B_MODEL_DIR)
print("MATRIX_GLOB:", MATRIX_GLOB)

if not STAGE03A_MODEL_DIR.is_dir():
    stage03a_root = REPO_ROOT / "outputs" / "stage03" / "03a"
    hint = ""
    if stage03a_root.is_dir():
        hint = "\nBuckets under outputs/stage03/03a/: " + ", ".join(
            sorted(p.name for p in stage03a_root.iterdir())
        )
    else:
        hint = "\noutputs/stage03/03a/ missing — run notebook 03a first."
    raise FileNotFoundError(f"03a bucket not found: {STAGE03A_MODEL_DIR}{hint}")

matrices = sorted(STAGE03A_MODEL_DIR.glob(MATRIX_GLOB))
if not matrices:
    raise FileNotFoundError(
        f"No files matching {MATRIX_GLOB} in {STAGE03A_MODEL_DIR}.\n"
        f"Example: sim_matrix_{GLOBAL_DESCRIPTOR}_bristol_gorillas_open.npy"
    )

print(f"Found {len(matrices)} open-set matrix file(s):")
for p in matrices:
    print(" ", p.name)


REPO_ROOT: C:\Users\Legion 5\Desktop\ApexID
STAGE03A_MODEL_DIR: C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03a\FineTuned_MiewID_DAF | exists: True
STAGE03B_MODEL_DIR: C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF
MATRIX_GLOB: sim_matrix_ApexID_Miewid_DAF_*_open.npy
Found 4 open-set matrix file(s):
  sim_matrix_ApexID_Miewid_DAF_bristol_gorillas_open.npy
  sim_matrix_ApexID_Miewid_DAF_elephants_open.npy
  sim_matrix_ApexID_Miewid_DAF_gcn_id_open.npy
  sim_matrix_ApexID_Miewid_DAF_jaguar_antonio_open.npy


In [2]:
%pip install -q optuna networkx scikit-learn

from collections import defaultdict
import hashlib
import json
import os
import random
from pathlib import Path

import networkx as nx
import numpy as np
import optuna
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.ERROR)
import pandas as pd
from sklearn.metrics import adjusted_rand_score, v_measure_score

# ============================================================
# Optuna knobs (paths: RUN TARGET cell above)
# ============================================================
_REQUIRED = ("STAGE03A_MODEL_DIR", "STAGE03B_MODEL_DIR", "REPO_ROOT", "GLOBAL_DESCRIPTOR", "MATRIX_GLOB")
_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the RUN TARGET cell first; missing: {_missing}")

# Full paper-style run: 2000. Smoke test: set env APEXID_OPTUNA_N_TRIALS=100 (or edit here).
N_TRIALS = int(os.environ.get("APEXID_OPTUNA_N_TRIALS", "2000"))
OPTUNA_SEED = 42
SPLIT_SEED = 42
KNOWN_IDENTITY_FRAC = 0.8
DB_FRAC_WITHIN_KNOWN = 0.8
MIN_IMAGES_PER_KNOWN_IDENTITY = 3
USE_RERANK = True

STAGE03_SPLITS_ROOT_CANDIDATES = (
    Path(os.environ.get("APEXID_STAGE03_SPLITS", "")),
    Path("/kaggle/input/datasets/mohamedbassat/03-open-set-splits-data/apexid_cvpr_stage03/03_splits"),
    Path("/kaggle/input/03-open-set-splits-data/apexid_cvpr_stage03/03_splits"),
    Path("/kaggle/working/apexid_cvpr_stage03/03_splits"),
)
QUERY_ONLY_DEFAULTS = {
    "cw_threshold": 0.30,
    "cw_top_k": 20,
    "cw_iterations": 20,
    "cw_k_core_degree": 1,
    "use_rerank": True,
    "rr_k1": 20,
    "rr_k2": 6,
    "rr_lambda": 0.30,
}

ALLOWED_DATASETS = ("bristol_gorillas", "spac_wild", "elephants", "gcn_id", "jaguar_antonio")


def parse_open_matrix_filename(path: Path) -> tuple[str, str]:
    name = path.name
    prefix, suffix = "sim_matrix_", "_open.npy"
    if not name.startswith(prefix) or not name.endswith(suffix):
        raise ValueError(f"Not an open-set matrix filename: {name}")
    middle = name[len(prefix) : -len(suffix)]
    for dataset in sorted(ALLOWED_DATASETS, key=len, reverse=True):
        token = f"_{dataset}"
        if middle.endswith(token):
            return middle[: -len(token)], dataset
    raise ValueError(f"Could not parse dataset from {name}; expected one of {ALLOWED_DATASETS}")


def discover_run_jobs(model_dir: Path, output_dir: Path, matrix_glob: str) -> list[dict]:
    jobs = []
    for matrix_path in sorted(model_dir.glob(matrix_glob)):
        descriptor_from_name, dataset_name = parse_open_matrix_filename(matrix_path)
        jobs.append(
            {
                "matrix_path": matrix_path.resolve(),
                "dataset_name": dataset_name,
                "descriptor_from_filename": descriptor_from_name,
                "output_root": output_dir.resolve(),
            }
        )
    if not jobs:
        raise FileNotFoundError(f"No sim_matrix_*_open.npy in {model_dir}")
    return jobs


RUN_JOBS = discover_run_jobs(STAGE03A_MODEL_DIR, STAGE03B_MODEL_DIR, MATRIX_GLOB)
print("RUN_JOBS:")
for job in RUN_JOBS:
    print(
        f"  - {job['dataset_name']}: {job['matrix_path'].name} -> {job['output_root']}"
    )
print(
    "STAGE03_SPLITS_ROOT:",
    next((p for p in STAGE03_SPLITS_ROOT_CANDIDATES if str(p) and p.is_dir()), "."),
)


Note: you may need to restart the kernel to use updated packages.


c:\Users\Legion 5\miniconda3\envs\crowd-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN_JOBS:
  - bristol_gorillas: sim_matrix_ApexID_Miewid_DAF_bristol_gorillas_open.npy -> C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF
  - elephants: sim_matrix_ApexID_Miewid_DAF_elephants_open.npy -> C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF
  - gcn_id: sim_matrix_ApexID_Miewid_DAF_gcn_id_open.npy -> C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF
  - jaguar_antonio: sim_matrix_ApexID_Miewid_DAF_jaguar_antonio_open.npy -> C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF
STAGE03_SPLITS_ROOT: .


In [3]:
def load_03a_open_bundle(matrix_path: Path, descriptor_from_filename: str, dataset_name: str):
    obj = np.load(matrix_path, allow_pickle=True).item()
    if dataset_name not in obj:
        raise KeyError(f"{matrix_path} has keys {list(obj)}; expected {dataset_name}")
    bundle = obj[dataset_name]
    global_descriptor = str(bundle.get("global_descriptor") or descriptor_from_filename)
    if bundle.get("dataset") != dataset_name:
        raise ValueError("03a dataset does not match filename")
    protocol = str(bundle.get("protocol") or "")
    if protocol not in {"gallery_attach_optuna", "query_only_defaults"}:
        raise ValueError(f"Unknown 03a protocol {protocol!r} in {matrix_path}")

    sim_qq = np.asarray(bundle["sim_query_query"], dtype=np.float32)
    sim_qg = np.asarray(
        bundle.get("sim_query_db", np.zeros((len(sim_qq), 0), dtype=np.float32)),
        dtype=np.float32,
    )
    query_image_ids = np.asarray(bundle["query_image_ids"]).astype(str)
    db_image_ids = np.asarray(bundle["db_image_ids"]).astype(str)
    y_query = np.asarray(bundle["query_identities"]).astype(str)
    y_db = np.asarray(bundle["db_identities"]).astype(str)
    query_open_role = np.asarray(bundle["query_open_role"]).astype(str)

    if sim_qq.shape != (len(query_image_ids), len(query_image_ids)):
        raise ValueError("Query-query matrix does not match query_image_ids")
    if sim_qg.shape != (len(query_image_ids), len(db_image_ids)):
        raise ValueError("Query-gallery matrix does not match query/gallery ids")
    if not np.isfinite(sim_qq).all() or sim_qq.min() < 0 or sim_qq.max() > 1:
        raise ValueError("Query-query similarities must be finite and clipped to [0,1]")
    if sim_qg.size and (not np.isfinite(sim_qg).all() or sim_qg.min() < 0 or sim_qg.max() > 1):
        raise ValueError("Query-gallery similarities must be finite and clipped to [0,1]")
    if len(set(query_image_ids)) != len(query_image_ids) or len(set(db_image_ids)) != len(db_image_ids):
        raise ValueError("03a image_ids are not unique")
    if set(query_image_ids) & set(db_image_ids):
        raise RuntimeError("Query/gallery image leakage in the 03a artifact")

    split = None
    if protocol == "gallery_attach_optuna":
        known_mask = query_open_role == "query_known"
        novel_mask = query_open_role == "query_novel"
        if not known_mask.any() or not novel_mask.any():
            raise RuntimeError("Open split must contain known and novel queries")
        if not len(db_image_ids):
            raise RuntimeError("Standard protocol requires a gallery")
        if set(y_query[novel_mask]) & set(y_db):
            raise RuntimeError("Unknown-query identities leaked into the gallery")
        for identity in np.unique(y_query[known_mask]):
            if identity not in set(y_db):
                raise RuntimeError(f"Known identity {identity!r} lacks a gallery sample")
        split = {
            "sim_query_query": sim_qq,
            "sim_query_db": sim_qg,
            "y_true_query": y_query,
            "y_true_db": y_db,
            "query_image_ids": query_image_ids,
            "db_image_ids": db_image_ids,
            "known_mask_query": known_mask,
            "unknown_mask_query": novel_mask,
        }
    else:
        if sim_qg.shape[1] != 0:
            raise RuntimeError("Gallery-free query-gallery matrix must be empty")

    return {
        "protocol": protocol,
        "global_descriptor": global_descriptor,
        "dataset_name": dataset_name,
        "matrix_path": matrix_path,
        "sim_qq": sim_qq,
        "sim_qg": sim_qg,
        "y_query": y_query,
        "y_db": y_db,
        "split": split,
    }


In [4]:
# AnimalCLEF identity scoring + graph clustering (ported from MIA-AI-Team/AnimalCLEF26).
def compute_identity_scores(sim_query_db, db_ids, top_r):
    identity_names = list(dict.fromkeys(db_ids.tolist()))
    scores = np.zeros((sim_query_db.shape[0], len(identity_names)), dtype=np.float32)
    for j, identity in enumerate(identity_names):
        cols = np.flatnonzero(db_ids == identity)
        sims = sim_query_db[:, cols]
        r = min(int(top_r), sims.shape[1])
        scores[:, j] = sims.mean(axis=1) if r == sims.shape[1] else np.partition(sims, -r, axis=1)[:, -r:].mean(axis=1)
    return np.asarray(identity_names, dtype=object), scores


def summarize_best_identities(names, scores):
    best_idx = scores.argmax(axis=1)
    best_scores = scores[np.arange(len(scores)), best_idx]
    best_names = names[best_idx]
    if scores.shape[1] >= 2:
        top2 = np.partition(scores, -2, axis=1)[:, -2:]
        second = np.min(top2, axis=1)
    else:
        second = np.zeros(len(scores), dtype=np.float32)
    return best_names, best_scores, second, best_scores - second


def get_topk_neighbors(sim_matrix, top_k):
    n = sim_matrix.shape[0]
    k = min(int(top_k), n - 1)
    if k <= 0:
        return np.empty((n, 0), dtype=np.int64)
    sims = sim_matrix.copy()
    np.fill_diagonal(sims, -np.inf)
    return np.argpartition(sims, -k, axis=1)[:, -k:]


def chinese_whispers(sim_matrix, threshold, iterations, top_k, k_core_degree, seed=42, skip_mutual_knn=False):
    random.seed(seed)
    np.random.seed(seed)
    n = sim_matrix.shape[0]
    adj = sim_matrix.copy()
    np.fill_diagonal(adj, 0)
    adj[adj < threshold] = 0
    graph = nx.from_numpy_array(adj)
    if graph.number_of_edges():
        try:
            backbone = nx.k_core(graph, k=int(k_core_degree))
            if backbone.number_of_nodes() == 0:
                backbone = graph
        except Exception:
            backbone = graph
    else:
        backbone = graph
    backbone_nodes = list(backbone.nodes()) or list(range(n))
    in_backbone = np.zeros(n, dtype=bool)
    in_backbone[backbone_nodes] = True
    valid = (sim_matrix >= threshold) & in_backbone[:, None] & in_backbone[None, :]
    np.fill_diagonal(valid, False)
    if not skip_mutual_knn:
        topk = get_topk_neighbors(sim_matrix, top_k)
        nearest = np.zeros((n, n), dtype=bool)
        if topk.shape[1]:
            nearest[np.arange(n)[:, None], topk] = True
        valid &= nearest & nearest.T
    sources, targets = np.nonzero(valid)
    graph_list = [[] for _ in range(n)]
    for source, target in zip(sources, targets):
        graph_list[int(source)].append((int(target), float(sim_matrix[source, target])))
    labels = list(range(n))
    nodes = backbone_nodes.copy()
    for _ in range(int(iterations)):
        random.shuffle(nodes)
        changes = 0
        for node in nodes:
            weights = defaultdict(float)
            for neighbor, weight in graph_list[node]:
                weights[labels[neighbor]] += weight
            if weights:
                new_label = max(weights.items(), key=lambda item: (item[1], -item[0]))[0]
                changes += int(new_label != labels[node])
                labels[node] = new_label
        if changes == 0:
            break
    return np.unique(labels, return_inverse=True)[1].astype(np.int64)


def final_open_set_labels(cluster_ids, best_names, best_scores, margins, tau_attach, tau_margin):
    votes = defaultdict(lambda: defaultdict(float))
    for i, cluster in enumerate(cluster_ids):
        if best_scores[i] >= tau_attach and margins[i] >= tau_margin:
            votes[int(cluster)][str(best_names[i])] += float(best_scores[i])
    cluster_to_known = {
        cluster: max(identity_votes.items(), key=lambda item: item[1])[0]
        for cluster, identity_votes in votes.items() if identity_votes
    }
    return np.asarray([
        cluster_to_known.get(int(cluster), f"__new_cluster_{int(cluster)}")
        for cluster in cluster_ids
    ], dtype=str)

In [5]:
# Exact k-reciprocal implementation used by the AnimalCLEF repository.
def re_ranking(sim_matrix, k1=20, k2=6, lambda_value=0.3):
    sim_matrix = np.asarray(sim_matrix, dtype=np.float32)
    n = sim_matrix.shape[0]
    original_dist = np.clip(1.0 - sim_matrix, 0.0, None)
    col_max = np.max(original_dist, axis=0)
    col_max[col_max == 0] = 1.0
    original_dist = (original_dist / col_max).T.astype(np.float32)
    values = np.zeros((n, n), dtype=np.float32)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)
    half_k1 = int(np.around(k1 / 2.0)) + 1
    for i in range(n):
        forward = initial_rank[i, : k1 + 1]
        backward = initial_rank[forward, : k1 + 1]
        reciprocal = forward[np.where(backward == i)[0]]
        expanded = reciprocal.copy()
        for q in reciprocal:
            q_forward = initial_rank[q, :half_k1]
            q_backward = initial_rank[q_forward, :half_k1]
            q_recip = q_forward[np.where(q_backward == q)[0]]
            if len(q_recip) and len(np.intersect1d(q_recip, reciprocal)) > (2.0 / 3.0) * len(q_recip):
                expanded = np.append(expanded, q_recip)
        expanded = np.unique(expanded)
        weight = np.exp(-original_dist[i, expanded])
        if weight.sum() > 0:
            values[i, expanded] = weight / weight.sum()
    if k2 > 1:
        expanded_values = np.zeros_like(values)
        for i in range(n):
            expanded_values[i] = np.mean(values[initial_rank[i, :k2]], axis=0)
        values = expanded_values
    jaccard = np.zeros((n, n), dtype=np.float32)
    inverse = [np.where(values[:, j] != 0)[0] for j in range(n)]
    for i in range(n):
        temp_min = np.zeros(n, dtype=np.float32)
        nonzero = np.where(values[i] != 0)[0]
        for column in nonzero:
            rows = inverse[column]
            temp_min[rows] += np.minimum(values[i, column], values[rows, column])
        jaccard[i] = 1.0 - temp_min / (2.0 - temp_min + 1e-12)
    final_dist = np.clip(jaccard * (1.0 - lambda_value) + original_dist * lambda_value, 0.0, 1.0)
    np.fill_diagonal(final_dist, 0.0)
    return (1.0 - final_dist).astype(np.float32)

In [6]:


class _DatasetOptunaProgress:
    """One tqdm bar per dataset (no per-trial Optuna INFO lines)."""

    def __init__(self, total: int, dataset_name: str, initial: int = 0) -> None:
        self._pbar = tqdm(
            total=total,
            initial=initial,
            desc=dataset_name,
            unit="trial",
            dynamic_ncols=True,
            leave=True,
        )

    def __call__(self, study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
        self._pbar.update(1)
        if study.best_trial is not None and study.best_value is not None:
            self._pbar.set_postfix(
                best=f"{study.best_value:.4f}",
                trial=study.best_trial.number,
                refresh=False,
            )

    def close(self) -> None:
        self._pbar.close()


def run_optuna_gallery_attach(split, output_root, global_descriptor, dataset_name):
    sim_qq = split["sim_query_query"]
    sim_qg = split["sim_query_db"]
    y_query = split["y_true_query"]
    y_db = split["y_true_db"]

    def objective(trial):
        top_r = trial.suggest_int("top_r", 1, 5)
        tau_attach = trial.suggest_float("tau_attach", 0.10, 0.90, step=0.01)
        tau_margin = trial.suggest_float("tau_margin", 0.10, 0.90, step=0.01)
        cw_threshold = trial.suggest_float("cw_threshold", 0.12, 0.68, step=0.01)
        cw_top_k = trial.suggest_int("cw_top_k", 7, 60)
        cw_iterations = trial.suggest_int("cw_iterations", 5, 30)
        cw_k_core_degree = trial.suggest_int("cw_k_core_degree", 1, 5)
        rr_k1 = trial.suggest_int("rr_k1", 5, 50)
        rr_k2 = trial.suggest_int("rr_k2", 1, 15)
        rr_lambda = trial.suggest_float("rr_lambda", 0.10, 0.90)

        identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, top_r)
        best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
        sim_for_cw = re_ranking(sim_qq, rr_k1, rr_k2, rr_lambda)
        clusters = chinese_whispers(
            sim_for_cw,
            threshold=cw_threshold,
            iterations=cw_iterations,
            top_k=cw_top_k,
            k_core_degree=cw_k_core_degree,
            seed=42,
            skip_mutual_knn=True,
        )
        predicted = final_open_set_labels(clusters, best_names, best_scores, margins, tau_attach, tau_margin)
        ari = adjusted_rand_score(y_query, predicted)
        v_measure = v_measure_score(y_query, predicted)
        trial.set_user_attr("ari", float(ari))
        trial.set_user_attr("v_measure", float(v_measure))
        return 0.6 * ari + 0.4 * v_measure

    storage_path = output_root / f"optuna_{global_descriptor}_{dataset_name}.sqlite3"
    study = optuna.create_study(
        study_name=f"{global_descriptor}__{dataset_name}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED, multivariate=True),
        storage=f"sqlite:///{storage_path.as_posix()}",
        load_if_exists=True,
    )
    completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
    remaining = max(0, N_TRIALS - completed_trials)
    if remaining:
        progress = _DatasetOptunaProgress(N_TRIALS, dataset_name, initial=completed_trials)
        try:
            study.optimize(
                objective,
                n_trials=remaining,
                gc_after_trial=True,
                show_progress_bar=False,
                callbacks=[progress],
            )
        finally:
            progress.close()
    else:
        tqdm.write(f"{dataset_name}: already complete ({completed_trials}/{N_TRIALS} trials)")
    print(
        f"  {dataset_name} done — best={study.best_value:.6f} "
        f"(trial {study.best_trial.number}, {completed_trials + remaining}/{N_TRIALS})"
    )
    return study


def freeze_config_for_job(loaded, study=None):
    protocol = loaded["protocol"]
    global_descriptor = loaded["global_descriptor"]
    dataset_name = loaded["dataset_name"]
    matrix_path = loaded["matrix_path"]
    output_root = loaded["output_root"]
    sim_qq = loaded["sim_qq"]
    sim_qg = loaded["sim_qg"]
    y_query = loaded["y_query"]
    y_db = loaded["y_db"]
    split = loaded["split"]

    split_sha256 = None
    winning_trial_recomputed = False

    if protocol == "gallery_attach_optuna":
        split_path = output_root / f"optuna_split_{global_descriptor}_{dataset_name}.npz"
        np.savez_compressed(split_path, **{k: v for k, v in split.items() if isinstance(v, np.ndarray)})
        split_sha256 = hashlib.sha256(split_path.read_bytes()).hexdigest()

        best_params = dict(study.best_params)
        best_params["use_rerank"] = USE_RERANK
        identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, int(best_params["top_r"]))
        best_names, best_scores, _, margins = summarize_best_identities(identity_names, identity_scores)
        best_sim_for_cw = re_ranking(
            sim_qq,
            int(best_params["rr_k1"]),
            int(best_params["rr_k2"]),
            float(best_params["rr_lambda"]),
        )
        best_clusters = chinese_whispers(
            best_sim_for_cw,
            threshold=float(best_params["cw_threshold"]),
            iterations=int(best_params["cw_iterations"]),
            top_k=int(best_params["cw_top_k"]),
            k_core_degree=int(best_params["cw_k_core_degree"]),
            seed=42,
            skip_mutual_knn=True,
        )
        best_predicted = final_open_set_labels(
            best_clusters,
            best_names,
            best_scores,
            margins,
            float(best_params["tau_attach"]),
            float(best_params["tau_margin"]),
        )
        recomputed_ari = float(adjusted_rand_score(y_query, best_predicted))
        recomputed_v_measure = float(v_measure_score(y_query, best_predicted))
        recomputed_objective = 0.6 * recomputed_ari + 0.4 * recomputed_v_measure
        if not np.isclose(recomputed_objective, float(study.best_value), rtol=0.0, atol=1e-12):
            raise RuntimeError(
                f"Winning trial not reproducible for {dataset_name}: "
                f"stored={study.best_value}, recomputed={recomputed_objective}"
            )
        completed_trials = sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)
        best_config = {
            **best_params,
            "protocol": protocol,
            "config_source": "optuna_stage03_open_split",
            "objective": "0.6*ARI+0.4*V-measure",
            "objective_value": recomputed_objective,
            "validation_ari": recomputed_ari,
            "validation_v_measure": recomputed_v_measure,
            "n_trials_completed": int(completed_trials),
            "split_seed": SPLIT_SEED,
            "known_identity_frac": KNOWN_IDENTITY_FRAC,
            "db_frac_within_known": DB_FRAC_WITHIN_KNOWN,
            "min_images_per_known_identity": MIN_IMAGES_PER_KNOWN_IDENTITY,
            "dataset": dataset_name,
            "global_descriptor": global_descriptor,
        }
        study.trials_dataframe().to_csv(
            output_root / f"optuna_trials_{global_descriptor}_{dataset_name}.csv", index=False
        )
        winning_trial_recomputed = True
    else:
        best_config = {
            **QUERY_ONLY_DEFAULTS,
            "protocol": protocol,
            "config_source": "frozen_label_free_defaults",
            "tuning_performed": False,
            "n_trials_completed": 0,
            "database_required": False,
            "known_attachment_enabled": False,
            "cw_seed": 42,
            "dataset": dataset_name,
            "global_descriptor": global_descriptor,
        }
        print("  Optuna skipped (query-only defaults):", QUERY_ONLY_DEFAULTS)

    config_path = output_root / f"best_configs_{global_descriptor}_{dataset_name}.json"
    config_path.write_text(json.dumps({dataset_name: best_config}, indent=2, sort_keys=True), encoding="utf-8")
    manifest = {
        "stage": "03b_frozen_clustering_config",
        "status": "passed",
        "protocol": protocol,
        "dataset": dataset_name,
        "global_descriptor": global_descriptor,
        "source_matrix": str(matrix_path),
        "source_matrix_sha256": hashlib.sha256(matrix_path.read_bytes()).hexdigest(),
        "optuna_split_sha256": split_sha256,
        "stage00_test_split_used_for_tuning": False,
        "stage03_open_split_used_for_tuning": protocol == "gallery_attach_optuna",
        "winning_trial_recomputed": winning_trial_recomputed,
        "rerank_mutual_knn_policy": "skip_mutual_knn=True after reranking",
        "config": best_config,
    }
    manifest_path = output_root / f"stage03b_manifest_{global_descriptor}_{dataset_name}.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    print("  Wrote:", config_path.name)
    return config_path


BATCH_SUMMARY = []
for job in RUN_JOBS:
    print("\n" + "=" * 72)
    print("JOB:", job["dataset_name"], "|", job["matrix_path"].name)
    loaded = load_03a_open_bundle(job["matrix_path"], job["descriptor_from_filename"], job["dataset_name"])
    loaded["output_root"] = job["output_root"]
    print("  protocol:", loaded["protocol"], "| descriptor:", loaded["global_descriptor"])
    if loaded["protocol"] == "gallery_attach_optuna":
        print(
            "  query-query:",
            loaded["sim_qq"].shape,
            "query-gallery:",
            loaded["sim_qg"].shape,
            "| gallery:",
            loaded["sim_qg"].shape[1],
        )
        study = run_optuna_gallery_attach(
            loaded["split"],
            job["output_root"],
            loaded["global_descriptor"],
            loaded["dataset_name"],
        )
    else:
        study = None
    config_path = freeze_config_for_job(loaded, study=study)
    BATCH_SUMMARY.append(
        {
            "dataset": loaded["dataset_name"],
            "global_descriptor": loaded["global_descriptor"],
            "protocol": loaded["protocol"],
            "config_path": str(config_path),
        }
    )

print("\n" + "=" * 72)
print(f"Done — {len(BATCH_SUMMARY)} dataset(s) in {STAGE03B_MODEL_DIR}")
for row in BATCH_SUMMARY:
    print(f"  {row['dataset']}: {row['protocol']} -> {row['config_path']}")
print("\nNEXT: attach 03b folder + 03a matrices to notebook 03c.")



JOB: bristol_gorillas | sim_matrix_ApexID_Miewid_DAF_bristol_gorillas_open.npy
  protocol: gallery_attach_optuna | descriptor: ApexID_Miewid_DAF
  query-query: (1789, 1789) query-gallery: (1789, 3535) | gallery: 3535


c:\Users\Legion 5\miniconda3\envs\crowd-gpu\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
bristol_gorillas: 100%|██████████| 2000/2000 [1:27:47<00:00,  2.63s/trial, best=0.2935, trial=1827]


  bristol_gorillas done — best=0.293487 (trial 1827, 2000/2000)


c:\Users\Legion 5\miniconda3\envs\crowd-gpu\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(


  Wrote: best_configs_ApexID_Miewid_DAF_bristol_gorillas.json

JOB: elephants | sim_matrix_ApexID_Miewid_DAF_elephants_open.npy
  protocol: gallery_attach_optuna | descriptor: ApexID_Miewid_DAF
  query-query: (821, 821) query-gallery: (821, 1230) | gallery: 1230


elephants: 100%|██████████| 2000/2000 [22:57<00:00,  1.45trial/s, best=0.4935, trial=793]


  elephants done — best=0.493545 (trial 793, 2000/2000)
  Wrote: best_configs_ApexID_Miewid_DAF_elephants.json

JOB: gcn_id | sim_matrix_ApexID_Miewid_DAF_gcn_id_open.npy
  protocol: gallery_attach_optuna | descriptor: ApexID_Miewid_DAF
  query-query: (428, 428) query-gallery: (428, 804) | gallery: 804


c:\Users\Legion 5\miniconda3\envs\crowd-gpu\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
gcn_id: 100%|██████████| 2000/2000 [17:55<00:00,  1.86trial/s, best=0.9615, trial=139]


  gcn_id done — best=0.961516 (trial 139, 2000/2000)
  Wrote: best_configs_ApexID_Miewid_DAF_gcn_id.json

JOB: jaguar_antonio | sim_matrix_ApexID_Miewid_DAF_jaguar_antonio_open.npy
  protocol: query_only_defaults | descriptor: ApexID_Miewid_DAF
  Optuna skipped (query-only defaults): {'cw_threshold': 0.3, 'cw_top_k': 20, 'cw_iterations': 20, 'cw_k_core_degree': 1, 'use_rerank': True, 'rr_k1': 20, 'rr_k2': 6, 'rr_lambda': 0.3}
  Wrote: best_configs_ApexID_Miewid_DAF_jaguar_antonio.json

Done — 4 dataset(s) in C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF
  bristol_gorillas: gallery_attach_optuna -> C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF\best_configs_ApexID_Miewid_DAF_bristol_gorillas.json
  elephants: gallery_attach_optuna -> C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03b\FineTuned_MiewID_DAF\best_configs_ApexID_Miewid_DAF_elephants.json
  gcn_id: gallery_attach_optuna -> C:\Users\Legion 5\Desktop\ApexID\outputs\stage0